<a href="https://colab.research.google.com/github/TimofeyProtasov/diploma/blob/main/days/work_2304.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q triton trl evaluate

In [2]:
import re
import torch
import random
import numpy as np
import pandas as pd
import time
import evaluate
from typing import Optional
from sklearn.model_selection import train_test_split
from dataclasses import dataclass, field
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from trl import SFTTrainer, SFTConfig
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftConfig
)
from tqdm import tqdm
from functools import wraps
from torch.utils.data import DataLoader

In [3]:
def measure_metrics(metric_name: str):
    """
    Декоратор для измерения времени выполнения и пиковой памяти GPU метода.
    Сохраняет результаты в self.metrics с ключами:
    - f"{metric_name}_time_min"
    - f"{metric_name}_peak_memory_gb"
    """
    def decorator(func):
        @wraps(func)
        def wrapper(self, *args, **kwargs):
            # Сбрасываем статистику пиковой памяти, если доступна CUDA
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()

            start_time = time.time()

            # Выполняем сам метод
            result = func(self, *args, **kwargs)

            elapsed_min = (time.time() - start_time) / 60.0
            self.metrics[f"{metric_name}_time_min"] = round(elapsed_min, 2)

            if torch.cuda.is_available():
                peak_memory_gb = torch.cuda.max_memory_allocated() / (1024**3)
                self.metrics[f"{metric_name}_peak_memory_gb"] = round(peak_memory_gb, 2)
            else:
                self.metrics[f"{metric_name}_peak_memory_gb"] = 0.0

            return result
        return wrapper
    return decorator

In [4]:
@dataclass
class RAGExperiment:
    dataset_name: str = 'phatvo/hotpotqa-raft'
    oracle_presence_ratio: float = 1.0
    train_size: int = 10
    test_size: int = 2
    model_name: str = 'Qwen/Qwen2.5-0.5B-Instruct'
    peft_config: Optional[PeftConfig] = field(default_factory=lambda: LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ))
    eval_epochs: list = field(default_factory=lambda: [3])
    per_device_train_batch_size: int = 4
    gradient_accumulation_steps: int = 2
    learning_rate: float = 5e-5
    warmup_steps: int = 5
    seed: int = 1

    def __post_init__(self):
        random.seed(self.seed)
        self.squad_metric = evaluate.load("squad")
        self.metrics = {}

        # Метрики качества
        self.f1_by_epoch = {}
        self.perplexity_by_epoch = {}

        # Время и память обучения
        self.train_time_by_epoch = {}
        self.train_memory_by_epoch = {}

        # Время и память оценки F1
        self.f1_time_by_epoch = {}
        self.f1_memory_by_epoch = {}

        # Время и память оценки перплексии
        self.perplexity_time_by_epoch = {}
        self.perplexity_memory_by_epoch = {}

    def prepare_data(self):
        dataset: DatasetDict = load_dataset('phatvo/hotpotqa-raft')


        def presence_oracle(example):
            return example['oracle_context'] in example['instruction']


        dataset_with_oracle: Dataset = dataset['train'].filter(lambda ex: presence_oracle(ex))
        dataset_without_oracle: Dataset = dataset['train'].filter(lambda ex: not presence_oracle(ex))

        self.sample_size = self.train_size + self.test_size

        num_with: int = round(self.sample_size * self.oracle_presence_ratio)
        num_without: int = round(self.sample_size * (1 - self.oracle_presence_ratio))

        if (num_with > len(dataset_with_oracle)) or (num_without > len(dataset_without_oracle)):
            raise ValueError("not enough samples in dataset")

        indices_with: list = random.sample(range(len(dataset_with_oracle)), num_with)
        size_train_with = round(self.train_size * self.oracle_presence_ratio)

        indices_without: list = random.sample(range(len(dataset_without_oracle)), num_without)
        size_train_without = round(self.train_size * (1 - self.oracle_presence_ratio))

        idx_train_with_oracle = indices_with[:size_train_with]
        idx_test_with_oracle = indices_with[size_train_with:num_with]

        idx_train_without_oracle = indices_without[:size_train_without]
        idx_test_without_oracle = indices_without[size_train_without:num_without]

        def get_shuffle_dataset(ids_with, ids_without):
            parts = []
            if ids_with:
                parts.append(dataset_with_oracle.select(ids_with))
            if ids_without:
                parts.append(dataset_without_oracle.select(ids_without))
            if parts:
                return concatenate_datasets(parts).shuffle(seed=self.seed)
            else:
                raise ValueError('empty sample')


        self.train_dataset: Dataset = get_shuffle_dataset(idx_train_with_oracle, idx_train_without_oracle)
        self.test_dataset: Dataset = get_shuffle_dataset(idx_test_with_oracle, idx_test_without_oracle)


    def prepare_model(self):
        attn = "eager" if isinstance(self.peft_config, PrefixTuningConfig) else "sdpa"
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            dtype=torch.bfloat16,
            device_map="auto",
            attn_implementation=attn,
        )

    def prepare_tokenizer(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token


    def format_data(self, dataset):
        """Форматируем данные в prompt/completion для SFTTrainer."""
        def format_sft_example(example):
            prompt_messages = [{"role": "user", "content": example["instruction"]}]
            prompt = self.tokenizer.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=True
            )
            completion = example["cot_answer"]
            return {"prompt": prompt, "completion": completion}

        return dataset.map(
            format_sft_example,
            remove_columns=dataset.column_names
        )


    def add_peft_if_exist(self):
        self.model = get_peft_model(self.model, self.peft_config) if self.peft_config else self.model
        self.metrics["trainable_params_m"] = self.model.num_parameters(only_trainable=True) / 1e6


    def prepare_training(self):
        total_epochs = self.eval_epochs[-1]   # последняя эпоха в списке
        self.training_args = SFTConfig(
            output_dir="./qwen-raft-lora",
            num_train_epochs=total_epochs,
            per_device_train_batch_size=self.per_device_train_batch_size,
            gradient_accumulation_steps=self.gradient_accumulation_steps,
            learning_rate=self.learning_rate,
            warmup_steps=self.warmup_steps,
            logging_steps=10,
            save_strategy="epoch",           # сохраняем после каждой эпохи
            save_steps=None,                 # отключаем шаговое сохранение
            save_total_limit=2,
            bf16=True,
            report_to="none",
            remove_unused_columns=False,

            max_length=1280,
            packing=False,
            completion_only_loss=True,
        )

        self.trainer = SFTTrainer(
            model=self.model,
            args=self.training_args,
            train_dataset=self.formatted_train_dataset,
            processing_class=self.tokenizer,
        )

    @measure_metrics("train")
    def train(self, resume: bool):
        self.trainer.train(resume_from_checkpoint=resume)


    def extract_answer(self, text: str) -> Optional[str]:
        """
        Извлекает ответ после <ANSWER>:.
        Поддерживает многострочные ответы до конца текста или до следующего маркера.
        """
        match = re.search(r"<ANSWER>:\s*(.*?)(?=\n<|$)", text, re.DOTALL)
        if match:
            return match.group(1).strip()
        return None


    @measure_metrics("evaluate_f1")
    def evaluate_f1(self, batch_size: int = 8):
        original_padding_side = self.tokenizer.padding_side
        self.tokenizer.padding_side = "left"

        def collate_fn(batch):
            prompts = [item["prompt"] for item in batch]
            completions = [item["completion"] for item in batch]
            true_answers = [self.extract_answer(c) for c in completions]

            tokenized = self.tokenizer(
                prompts,
                padding=True,
                truncation=True,
                max_length=self.training_args.max_length,
                return_tensors="pt"
            )
            return {
                "prompts": prompts,
                "input_ids": tokenized["input_ids"],
                "attention_mask": tokenized["attention_mask"],
                "true_answers": true_answers,
                "ids": [str(i) for i in range(len(batch))]
            }

        dataloader = DataLoader(
            self.formatted_test_dataset,
            batch_size=batch_size,
            collate_fn=collate_fn,
            shuffle=False
        )

        f1_scores = []

        for batch in tqdm(dataloader, desc=f"Evaluating F1 (batch_size={batch_size})"):
            input_ids = batch["input_ids"].to(self.model.device)
            attention_mask = batch["attention_mask"].to(self.model.device)

            with torch.no_grad():
                outputs = self.model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=1024,
                    do_sample=False,
                    pad_token_id=self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                    temperature=None,
                    top_p=None,
                    top_k=None,
                )

            for i, (prompt, true_answer, example_id) in enumerate(zip(
                batch["prompts"], batch["true_answers"], batch["ids"]
            )):
                if true_answer is None:
                    continue

                prompt_len = attention_mask[i].sum().item()  # реальная длина промпта
                generated_ids = outputs[i][prompt_len:]
                generated_part = self.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
                pred_answer = self.extract_answer(generated_part)

                if pred_answer is None:
                    f1 = 0.0
                else:
                    pred_dict = {"prediction_text": pred_answer, "id": example_id}
                    ref_dict = {"answers": {"answer_start": [0], "text": [true_answer]}, "id": example_id}
                    result = self.squad_metric.compute(predictions=[pred_dict], references=[ref_dict])
                    f1 = result["f1"] / 100.0

                f1_scores.append(f1)

        self.tokenizer.padding_side = original_padding_side

        avg_f1 = np.mean(f1_scores) if f1_scores else 0.0
        self.metrics["f1"] = round(float(avg_f1), 2)


    @measure_metrics("evaluate_perplexity")
    def evaluate_perplexity(self):
        """
        Вычисляет перплексию на тестовом наборе без батчинга.
        Потери считаются только по completion.
        """
        total_loss = 0.0
        total_tokens = 0

        for example in tqdm(self.formatted_test_dataset, desc="Calculating perplexity"):
            prompt = example["prompt"]
            completion = example["completion"]
            full_text = prompt + completion

            # Токенизируем полный текст (с параметрами обучения)
            tokenized_full = self.tokenizer(
                full_text,
                truncation=True,
                max_length=self.training_args.max_length,
                return_tensors="pt"
            )
            input_ids = tokenized_full["input_ids"][0]

            # Токенизируем ТОЛЬКО промпт с теми же параметрами, чтобы узнать его длину
            tokenized_prompt = self.tokenizer(
                prompt,
                truncation=True,
                max_length=self.training_args.max_length,
                return_tensors="pt"
            )
            prompt_len = tokenized_prompt["input_ids"].shape[1]

            # Если полный текст был обрезан и промпт занял всё окно,
            # то completion мог не попасть вообще. В таком случае пропускаем пример.
            if prompt_len >= len(input_ids):
                continue

            # Создаём labels: копируем input_ids, маскируем промпт
            labels = input_ids.clone()
            labels[:prompt_len] = -100

            # Переносим на устройство
            input_ids = input_ids.unsqueeze(0).to(self.model.device)
            labels = labels.unsqueeze(0).to(self.model.device)
            attention_mask = torch.ones_like(input_ids).to(self.model.device)

            with torch.no_grad():
                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss

            num_tokens = (labels != -100).sum().item()
            if num_tokens > 0:
                total_loss += loss.item() * num_tokens
                total_tokens += num_tokens

        avg_loss = total_loss / total_tokens if total_tokens > 0 else float("inf")
        perplexity = np.exp(avg_loss)

        self.metrics["perplexity"] = round(float(perplexity), 2)


    def run(self):
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        self.prepare_data()
        self.prepare_model()
        self.prepare_tokenizer()
        self.formatted_train_dataset = self.format_data(self.train_dataset)
        self.formatted_test_dataset = self.format_data(self.test_dataset)
        self.add_peft_if_exist()
        self.prepare_training()
        self.metrics["before_train_peak_memory_gb"] = round(torch.cuda.max_memory_allocated() / (1024**3), 2)

        current_epoch = 0
        for target_epoch in self.eval_epochs:
            if target_epoch <= current_epoch:
                continue

            self.training_args.num_train_epochs = target_epoch
            resume = (current_epoch > 0)

            # Обучение
            self.train(resume)
            current_epoch = target_epoch

            self.train_time_by_epoch[current_epoch] = self.metrics.get("train_time_min", 0.0)
            self.train_memory_by_epoch[current_epoch] = self.metrics.get("train_peak_memory_gb", 0.0)

            self.model.eval()

            # Оценка F1
            self.evaluate_f1(batch_size=8)
            self.f1_by_epoch[current_epoch] = self.metrics["f1"]
            self.f1_time_by_epoch[current_epoch] = self.metrics.get("evaluate_f1_time_min", 0.0)
            self.f1_memory_by_epoch[current_epoch] = self.metrics.get("evaluate_f1_peak_memory_gb", 0.0)

            # Оценка перплексии
            self.evaluate_perplexity()
            self.perplexity_by_epoch[current_epoch] = self.metrics["perplexity"]
            self.perplexity_time_by_epoch[current_epoch] = self.metrics.get("evaluate_perplexity_time_min", 0.0)
            self.perplexity_memory_by_epoch[current_epoch] = self.metrics.get("evaluate_perplexity_peak_memory_gb", 0.0)

        # Собираем всё в общий словарь метрик
        self.metrics["f1_by_epoch"] = self.f1_by_epoch
        self.metrics["perplexity_by_epoch"] = self.perplexity_by_epoch
        self.metrics["train_time_by_epoch"] = self.train_time_by_epoch
        self.metrics["train_memory_by_epoch"] = self.train_memory_by_epoch
        self.metrics["f1_time_by_epoch"] = self.f1_time_by_epoch
        self.metrics["f1_memory_by_epoch"] = self.f1_memory_by_epoch
        self.metrics["perplexity_time_by_epoch"] = self.perplexity_time_by_epoch
        self.metrics["perplexity_memory_by_epoch"] = self.perplexity_memory_by_epoch

In [5]:
from peft import (
    LoraConfig,
    BOFTConfig,
    PrefixTuningConfig,
    IA3Config,
    TaskType
)

# 1. LoRA — базовый (низкоранговая перепараметризация)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# 2. BOFT — ортогональная тонкая настройка (принципиально иной подход)
boft_config = BOFTConfig(
    boft_block_size=4,
    boft_n_butterfly_factor=1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    boft_dropout=0.0,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# 3. Prefix Tuning — мягкие промпты ко всем слоям
prefix_config = PrefixTuningConfig(
    num_virtual_tokens=10,
    encoder_hidden_size=896,       # размерность скрытого слоя Qwen2.5‑0.5B
    prefix_projection=True,        # стабилизирует обучение (P‑Tuning v2)
    task_type=TaskType.CAUSAL_LM,
)

# 4. IA³ — масштабирование ключей, значений и FFN
ia3_config = IA3Config(
    target_modules=["k_proj", "v_proj", "down_proj"],
    feedforward_modules=["down_proj"],
    task_type=TaskType.CAUSAL_LM,
)

In [6]:
exp31 = RAGExperiment(
    dataset_name='phatvo/hotpotqa-raft',
    oracle_presence_ratio=1.0,
    train_size=180,
    test_size=20,
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    peft_config=prefix_config,
    eval_epochs=[3],
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=1
)

In [7]:
exp31.run()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/180 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/180 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.


Step,Training Loss
10,0.882710
20,0.903142
30,0.893486
40,0.904139
50,0.903046
60,0.877523


Evaluating F1 (batch_size=8):   0%|          | 0/3 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")
Calculating perplexity: 100%|██████████| 20/20 [00:01<00:00, 15.32it/s]


In [8]:
exp31.metrics

{'trainable_params_m': 0.0,
 'before_train_peak_memory_gb': 1.43,
 'train_time_min': 2.73,
 'train_peak_memory_gb': 9.89,
 'f1': 0.0,
 'evaluate_f1_time_min': 1.85,
 'evaluate_f1_peak_memory_gb': 1.79,
 'perplexity': 4.43,
 'evaluate_perplexity_time_min': 0.02,
 'evaluate_perplexity_peak_memory_gb': 3.19,
 'f1_by_epoch': {3: 0.0},
 'perplexity_by_epoch': {3: 4.43},
 'train_time_by_epoch': {3: 2.73},
 'train_memory_by_epoch': {3: 9.89},
 'f1_time_by_epoch': {3: 1.85},
 'f1_memory_by_epoch': {3: 1.79},
 'perplexity_time_by_epoch': {3: 0.02},
 'perplexity_memory_by_epoch': {3: 3.19}}

In [9]:
del exp31

In [10]:
exp32 = RAGExperiment(
    dataset_name='phatvo/hotpotqa-raft',
    oracle_presence_ratio=1.0,
    train_size=270,
    test_size=20,
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    peft_config=prefix_config,
    eval_epochs=[7],
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=1
)

In [11]:
exp32.run()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map:   0%|          | 0/270 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/270 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/270 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.898868
20,0.878187
30,0.858801
40,0.887961
50,0.874875
60,0.876467
70,0.876010
80,0.852765
90,0.920616
100,0.876580


Evaluating F1 (batch_size=8):   0%|          | 0/3 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")
Calculating perplexity: 100%|██████████| 20/20 [00:01<00:00, 14.92it/s]


In [12]:
exp32.metrics

{'trainable_params_m': 0.0,
 'before_train_peak_memory_gb': 2.39,
 'train_time_min': 9.3,
 'train_peak_memory_gb': 9.89,
 'f1': 0.0,
 'evaluate_f1_time_min': 1.86,
 'evaluate_f1_peak_memory_gb': 1.95,
 'perplexity': 4.72,
 'evaluate_perplexity_time_min': 0.02,
 'evaluate_perplexity_peak_memory_gb': 3.08,
 'f1_by_epoch': {7: 0.0},
 'perplexity_by_epoch': {7: 4.72},
 'train_time_by_epoch': {7: 9.3},
 'train_memory_by_epoch': {7: 9.89},
 'f1_time_by_epoch': {7: 1.86},
 'f1_memory_by_epoch': {7: 1.95},
 'perplexity_time_by_epoch': {7: 0.02},
 'perplexity_memory_by_epoch': {7: 3.08}}

In [13]:
del exp32

In [14]:
exp21 = RAGExperiment(
    dataset_name='phatvo/hotpotqa-raft',
    oracle_presence_ratio=1.0,
    train_size=180,
    test_size=20,
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    peft_config=boft_config,
    eval_epochs=[3],
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=1
)
exp21.run()
print(exp21.metrics)
del exp21

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/boft/layer.py:95: UserWarning: Failed to load the CUDA extension: Ninja is required to load C++ extensions (pip install ninja to get it), check if ninja is available.
  warnings.warn(f"Failed to load the CUDA extension: {e}, check if ninja is available.")
/usr/local/lib/python3.12/dist-packages/peft/tuners/boft/layer.py:96: UserWarning: Setting boft_n_butterfly_factor to 1 to speed up the finetuning process.
  warnings.warn("Setting boft_n_butterfly_factor to 1 to speed up the finetuning process.")
/usr/local/lib/python3.12/dist-packages/peft/tuners/boft/layer.py:95: UserWarning: Failed to load the CUDA extension: /root/.cache/torch_extensions/py312_cu128/fbd_cuda/fbd_cuda.so: cannot open shared object file: No such file or directory, check if ninja is available.
  warnings.warn(f"Failed to load the CUDA extension: {e}, check if ninja is available.")


Adding EOS to train dataset:   0%|          | 0/180 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/180 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.115011
20,1.027437
30,0.951251
40,0.887094
50,0.864737
60,0.818428


Calculating perplexity: 100%|██████████| 20/20 [00:17<00:00,  1.14it/s]

{'trainable_params_m': 1.287168, 'before_train_peak_memory_gb': 3.49, 'train_time_min': 6.22, 'train_peak_memory_gb': 12.43, 'f1': 0.0, 'evaluate_f1_time_min': 12.07, 'evaluate_f1_peak_memory_gb': 3.91, 'perplexity': 2.21, 'evaluate_perplexity_time_min': 0.29, 'evaluate_perplexity_peak_memory_gb': 5.7, 'f1_by_epoch': {3: 0.0}, 'perplexity_by_epoch': {3: 2.21}, 'train_time_by_epoch': {3: 6.22}, 'train_memory_by_epoch': {3: 12.43}, 'f1_time_by_epoch': {3: 12.07}, 'f1_memory_by_epoch': {3: 3.91}, 'perplexity_time_by_epoch': {3: 0.29}, 'perplexity_memory_by_epoch': {3: 5.7}}


In [15]:
exp22 = RAGExperiment(
    dataset_name='phatvo/hotpotqa-raft',
    oracle_presence_ratio=1.0,
    train_size=270,
    test_size=20,
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    peft_config=boft_config,
    eval_epochs=[7],
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=1
)
exp22.run()
print(exp22.metrics)
del exp22

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map:   0%|          | 0/270 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/boft/layer.py:95: UserWarning: Failed to load the CUDA extension: /root/.cache/torch_extensions/py312_cu128/fbd_cuda/fbd_cuda.so: cannot open shared object file: No such file or directory, check if ninja is available.
  warnings.warn(f"Failed to load the CUDA extension: {e}, check if ninja is available.")
/usr/local/lib/python3.12/dist-packages/peft/tuners/boft/layer.py:96: UserWarning: Setting boft_n_butterfly_factor to 1 to speed up the finetuning process.
  warnings.warn("Setting boft_n_butterfly_factor to 1 to speed up the finetuning process.")


Adding EOS to train dataset:   0%|          | 0/270 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/270 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.149147
20,1.016674
30,0.904650
40,0.833305
50,0.811673
60,0.740768
70,0.723553
80,0.678992
90,0.662414
100,0.642888


Calculating perplexity: 100%|██████████| 20/20 [00:17<00:00,  1.13it/s]

{'trainable_params_m': 1.287168, 'before_train_peak_memory_gb': 4.92, 'train_time_min': 21.78, 'train_peak_memory_gb': 12.43, 'f1': 0.15, 'evaluate_f1_time_min': 11.51, 'evaluate_f1_peak_memory_gb': 3.94, 'perplexity': 1.65, 'evaluate_perplexity_time_min': 0.29, 'evaluate_perplexity_peak_memory_gb': 5.59, 'f1_by_epoch': {7: 0.15}, 'perplexity_by_epoch': {7: 1.65}, 'train_time_by_epoch': {7: 21.78}, 'train_memory_by_epoch': {7: 12.43}, 'f1_time_by_epoch': {7: 11.51}, 'f1_memory_by_epoch': {7: 3.94}, 'perplexity_time_by_epoch': {7: 0.29}, 'perplexity_memory_by_epoch': {7: 5.59}}


In [16]:
exp41 = RAGExperiment(
    dataset_name='phatvo/hotpotqa-raft',
    oracle_presence_ratio=1.0,
    train_size=180,
    test_size=20,
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    peft_config=ia3_config,
    eval_epochs=[3],
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=1
)
exp41.run()
print(exp41.metrics)
del exp41

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/180 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/180 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.124333
20,1.113930
30,1.121173
40,1.115180
50,1.125437
60,1.103476


Calculating perplexity: 100%|██████████| 20/20 [00:01<00:00, 17.80it/s]

{'trainable_params_m': 0.12288, 'before_train_peak_memory_gb': 4.92, 'train_time_min': 1.82, 'train_peak_memory_gb': 9.87, 'f1': 0.0, 'evaluate_f1_time_min': 0.22, 'evaluate_f1_peak_memory_gb': 1.41, 'perplexity': 2.92, 'evaluate_perplexity_time_min': 0.02, 'evaluate_perplexity_peak_memory_gb': 3.14, 'f1_by_epoch': {3: 0.0}, 'perplexity_by_epoch': {3: 2.92}, 'train_time_by_epoch': {3: 1.82}, 'train_memory_by_epoch': {3: 9.87}, 'f1_time_by_epoch': {3: 0.22}, 'f1_memory_by_epoch': {3: 1.41}, 'perplexity_time_by_epoch': {3: 0.02}, 'perplexity_memory_by_epoch': {3: 3.14}}


In [17]:
exp42 = RAGExperiment(
    dataset_name='phatvo/hotpotqa-raft',
    oracle_presence_ratio=1.0,
    train_size=270,
    test_size=20,
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    peft_config=ia3_config,
    eval_epochs=[7],
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=1
)
exp42.run()
print(exp42.metrics)
del exp42

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map:   0%|          | 0/270 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/270 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/270 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.159941
20,1.109424
30,1.081799
40,1.077562
50,1.131580
60,1.065813
70,1.088730
80,1.035724
90,1.049806
100,1.023011


Calculating perplexity: 100%|██████████| 20/20 [00:01<00:00, 17.59it/s]

{'trainable_params_m': 0.12288, 'before_train_peak_memory_gb': 2.37, 'train_time_min': 6.21, 'train_peak_memory_gb': 9.87, 'f1': 0.0, 'evaluate_f1_time_min': 0.21, 'evaluate_f1_peak_memory_gb': 1.46, 'perplexity': 2.63, 'evaluate_perplexity_time_min': 0.02, 'evaluate_perplexity_peak_memory_gb': 3.03, 'f1_by_epoch': {7: 0.0}, 'perplexity_by_epoch': {7: 2.63}, 'train_time_by_epoch': {7: 6.21}, 'train_memory_by_epoch': {7: 9.87}, 'f1_time_by_epoch': {7: 0.21}, 'f1_memory_by_epoch': {7: 1.46}, 'perplexity_time_by_epoch': {7: 0.02}, 'perplexity_memory_by_epoch': {7: 3.03}}


In [ ]:
import os
import pandas as pd
from pathlib import Path
import torch
import gc

# Монтируем Google Drive (если ещё не)
from google.colab import drive
drive.mount('/content/drive')

# Путь для сохранения CSV
SAVE_DIR = "/content/drive/MyDrive/rag_experiments"
os.makedirs(SAVE_DIR, exist_ok=True)

# Параметры экспериментов
train_sizes = [100, 300, 500, 700]
test_size = 50
eval_epochs = [4, 8, 12]

def flatten_metrics(metrics: dict, train_size: int) -> dict:
    """Преобразует вложенные словари метрик в плоский словарь для одной строки CSV."""
    flat = {"train_size": train_size}
    for key, value in metrics.items():
        if isinstance(value, dict):
            # Например, "f1_by_epoch": {5: 0.5, 10: 0.6, ...}
            for epoch, val in value.items():
                # Убираем суффикс "_by_epoch" и добавляем номер эпохи
                base_key = key.replace("_by_epoch", "")
                flat[f"{base_key}_epoch{epoch}"] = val
        else:
            flat[key] = value
    return flat


Mounted at /content/drive


In [ ]:
# Основной цикл
for train_size in train_sizes:
    csv_path = os.path.join(SAVE_DIR, f"results_train_{train_size}.csv")

    # Проверяем, не сохранён ли уже результат
    if os.path.exists(csv_path):
        print(f"Результаты для train_size={train_size} уже существуют, пропускаем.")
        continue

    print(f"\n{'='*50}")
    print(f"Запуск эксперимента для train_size={train_size}")
    print(f"{'='*50}")

    try:
        # Создаём и запускаем эксперимент
        exp = RAGExperiment(
            train_size=train_size,
            test_size=test_size,
            eval_epochs=eval_epochs,
            # Остальные параметры можно оставить по умолчанию или задать явно
        )
        exp.run()

        # Формируем плоский словарь и сохраняем в CSV
        flat_metrics = flatten_metrics(exp.metrics, train_size)
        df = pd.DataFrame([flat_metrics])
        df.to_csv(csv_path, index=False)
        print(f"Результаты сохранены в {csv_path}")

    except Exception as e:
        print(f"Ошибка при обработке train_size={train_size}: {e}")
        # При желании можно сохранить информацию об ошибке и продолжить
        continue

    finally:
        # Освобождаем ресурсы
        del exp
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()


Запуск эксперимента для train_size=100


README.md:   0%|          | 0.00/651 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.30M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1855 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1855 [00:00<?, ? examples/s]

...

Step,Training Loss
360,0.462131
370,0.467296
380,0.457977
390,0.465288
400,0.489811
410,0.474359
420,0.485165
430,0.463172
440,0.458699
450,0.472909


Calculating perplexity: 100%|██████████| 50/50 [00:03<00:00, 15.47it/s]


Step,Training Loss
710,0.434969
720,0.430549
730,0.457141
740,0.443366
750,0.445695
760,0.463448
770,0.424714
780,0.430457
790,0.445783
800,0.455666


Calculating perplexity: 100%|██████████| 50/50 [00:03<00:00, 15.49it/s]


Результаты сохранены в /content/drive/MyDrive/rag_experiments/results_train_700.csv


In [ ]:
def load_all_results(save_dir: str) -> pd.DataFrame:
    all_files = Path(save_dir).glob("results_train_*.csv")
    df_list = []
    for file in all_files:
        df = pd.read_csv(file)
        df_list.append(df)
    if df_list:
        return pd.concat(df_list, ignore_index=True)
    else:
        return pd.DataFrame()

df_all = load_all_results(SAVE_DIR)
print(df_all)

   train_size  trainable_params_m  before_train_peak_memory_gb  \
0         100            0.540672                         1.43   
1         300            0.540672                         1.45   
2         500            0.540672                         1.45   
3         700            0.540672                         1.45   

   train_time_min  train_peak_memory_gb    f1  evaluate_f1_time_min  \
0            1.17                  9.87  0.00                  1.82   
1            3.45                  9.87  0.30                  1.87   
2            5.71                  9.87  0.55                  1.56   
3            8.07                  9.87  0.50                  1.64   

   evaluate_f1_peak_memory_gb  perplexity  evaluate_perplexity_time_min  ...  \
0                        1.34        2.18                          0.05  ...   
1                        1.32        1.74                          0.05  ...   
2                        1.30        1.53                          0.05  